In [23]:
import numpy as np
import requests
import nltk
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import re
from collections import Counter, OrderedDict


nltk.download('punkt')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nikhi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [24]:
url = "https://www.gutenberg.org/cache/epub/7849/pg7849.txt"
response = requests.get(url)
text = response.text

start_idx = text.find("*** START OF THE PROJECT GUTENBERG EBOOK THE TRIAL ***")
end_idx = text.find("End of the Project Guetnberg")
text = text[start_idx:end_idx]

In [32]:
class TokenEncoder:
    def __init__(self, corpus: str, seq_length: int, chunk_size: int) -> None:
        self.tokens = self.tokenizer(corpus)
        self.token_set = sorted(set(self.tokens))
        self.token2int = {tok: i for i, tok in enumerate(self.token_set)}
        self.tokens_array = np.array(self.token_set)
        self.tokens_encoded = np.array(
            [self.token2int[tok] for tok in self.tokens]
        )
        self.chunks = [
            self.tokens_encoded[i:i+seq_length+chunk_size] 
            for i in range(len(self.tokens_encoded)-seq_length-chunk_size+1)
        ]
        
    def encode(self, s: str) -> np.ndarray:
        tokens = self.tokenizer(s)
        return np.array([self.token2int[tok] for tok in tokens if tok in self.token2int])

    def decode(self, indices: np.ndarray) -> list:
        return [self.tokens_array[i] for i in indices if i < len(self.tokens_array)]
    
    @staticmethod
    def tokenizer(s: str) -> list:  
        s = re.sub("<[^>]*>", "", s)
        tokenized = s.split()
        return tokenized

In [35]:
token_encoder = TokenEncoder(text, 100, 1)
print(f"Total number of tokens: {len(token_encoder.tokens)}")
print(f"Number of unique tokens: {len(token_encoder.token_set)}")

Total number of tokens: 86713
Number of unique tokens: 9411


In [37]:
new_encoded = token_encoder.encode("K. cried in the court of law.")
print(new_encoded)

decoded = token_encoder.decode(new_encoded)
print(" ".join(decoded))

[ 713 2744 4776 8299 2697 6062 5262]
K. cried in the court of law.
